# 📰 MINI PROJECT: TEXT SUMMARIZATION BAHASA INDONESIA
## Komparasi Extractive Summarization: TF-IDF Sentence Ranking vs. TextRank Algorithm
**Tugas Lanjutan Pertemuan 11 & 12 — Pemrosesan Bahasa Alami (NLP)**  
**Program Studi:** S1 Teknik Informatika — Universitas Negeri Malang (UM)  
**Dosen Pengampu:** Prof. Ir. Aji Prasetya Wibawa, S.T., M.M.T., Ph.D. & M. Zainal Arifin, S.Si., M.Kom., Ph.D.  
**Mentor Asisten:** Agung Bella Putra Utama & Fadia Irsania Putri  

---

### 👥 Pembagian Peran & Job Desk Tim (5 Anggota)

| No | NIM | Nama Mahasiswa | Peran & Tanggung Jawab Utama |
| :-: | :---: | :--- | :--- |
| **1** | `240535600363` | **ASYAM SAKHI MAULANA** | **Pengembang Model 1 (TF-IDF Sentence Ranking):**<br>• Preprocessing internal: segmentasi kalimat & normalisasi teks artikel.<br>• Pembentukan matriks TF-IDF kalimat & pembobotan salience score.<br>• Seleksi Top-K kalimat penting & rekonstruksi ringkasan kronologis.<br>• Pengujian & analisis karakteristik inferensi Model 1. |
| **2** | `240535607403` | **AGUSTINUS MARIA AGUNG HERIYANTO** | **Pengembang Model 2 (TextRank Algorithm):**<br>• Preprocessing internal & segmentasi kalimat untuk graf.<br>• Konstruksi matriks kemiripan kosinus antar-kalimat & graf NetworkX.<br>• Kalkulasi sentralitas PageRank & rekonstruksi ringkasan TextRank.<br>• Pengujian inferensi Model 2. |
| **3** | `240535604624` | **CHRISTIAN LIONEL DAUN DARMA** | **Dataset Preparation, Evaluator ROUGE & Kesimpulan:**<br>• Pemuatan & kurasi dataset benchmark IndoSum resmi (Kata.ai).<br>• Eksplorasi statistik korpus berita (panjang kata, rasio kompresi).<br>• Evaluasi komparatif metrik ROUGE (ROUGE-1, ROUGE-2, ROUGE-L).<br>• Analisis kualitatif ringkasan terbaik/terburuk & penulisan kesimpulan komparatif. |
| **4** | `240535601034` | **ALBI ADWITIYA CHANDRA** | **Mini Project Text Generation:** Pengembang Model 1 (LSTM Language Model). |
| **5** | `240535610197` | **INDAH HARIYANTI** | **Mini Project Text Generation:** Pengembang Model 2 (GRU Language Model). |

---

### 🎯 Tujuan & Aspek Komparasi Sesuai Slide 20 PDF
Masing-masing kelompok wajib membandingkan kedua model ekstraktif dari sisi:
1. **Skor ROUGE-1, ROUGE-2, dan ROUGE-L** terhadap ringkasan acuan manusia (*gold summary*)
2. **Kualitas ringkasan** dan kelancaran alur baca (*coherence*)
3. **Kemampuan mempertahankan informasi penting** (*salience preservation*)
4. **Konsistensi pemilihan kalimat** dan eliminasi redundansi
5. **Waktu proses komputasi** (*processing latency*)
6. **Contoh ringkasan terbaik dan terburuk**
7. **Analisis error & kelemahan masing-masing metode**
8. **Diskusi Pertanyaan Utama:** *Apakah model dengan skor ROUGE tertinggi selalu menghasilkan ringkasan yang paling baik secara makna?*
9. **Bonus:** Komparasi hasil extractive summarization dengan model abstractive summarization.


## 1. Setup Lingkungan & Import Library
Instalasi pustaka `rouge-score` dan `networkx` untuk evaluasi metrik peringkasan dan pemodelan graf.

In [ ]:
# Instalasi library pendukung jika belum tersedia di runtime
!pip install --quiet rouge-score datasets networkx Sastrawi

import os
import re
import time
import math
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# Konfigurasi tampilan
pd.set_option('display.max_colwidth', 150)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
SEED = 42
np.random.seed(SEED)

print("Setup lingkungan berhasil!")
print("Library NLP & Evaluasi siap digunakan.")

---
## 2. Dataset IndoSum: Pemuatan & Eksplorasi Korpus Berita
**Penanggung Jawab:** **CHRISTIAN LIONEL DAUN DARMA** (NIM: `240535604624`)  

Dataset yang digunakan adalah **IndoSum (Kata.ai)** (`https://github.com/kata-ai/indosum`), korpus benchmark standar untuk tugas *Text Summarization* bahasa Indonesia yang dikurasi dari berbagai portal berita nasional (CNN Indonesia, Detik, Kumparan, dll). Dataset ini menyediakan artikel berita lengkap beserta ringkasan acuan manusia (*gold summary*).

> **📌 Catatan Teknis Pemuatan Dataset:**
> Skrip pemuat di bawah dirancang fleksibel:
> 1. Jika file resmi IndoSum (`test.01.jsonl` atau `train.01.jsonl`) diunggah/tersedia di Google Colab, skrip akan langsung membaca file resmi tersebut secara otomatis.
> 2. Jika dijalankan secara mandiri tanpa file lokal (misal saat demo atau kuota terbatas), skrip otomatis menggunakan koleksi benchmark IndoSum teranotasi bawaan agar **notebook selalu siap dieksekusi tanpa potensi crash**.


In [ ]:
# ==============================================================================
# BAGIAN 2.1: PEMUATAN DATASET INDOSUM RESMI (KATA.AI)
# Pengembang: Christian Lionel Daun Darma (240535604624)
# ==============================================================================

import os
import json
import pandas as pd

def load_indosum_corpus(jsonl_file=None, max_samples=50):
    """
    Memuat korpus IndoSum dari Kata.ai (format .jsonl) atau menggunakan 
    koleksi benchmark teranotasi bawaan jika file lokal tidak ditemukan.
    
    Format standar IndoSum:
    - paragraphs: List of paragraphs -> List of sentences -> List of words
    - summary: List of sentences -> List of words
    """
    # 1. Deteksi file JSONL resmi jika tersedia di direktori kerja / Colab
    target_files = [jsonl_file, "test.01.jsonl", "train.01.jsonl", "dev.01.jsonl"]
    found_file = next((f for f in target_files if f and os.path.exists(f)), None)
    
    if found_file:
        print(f"📥 Memuat dataset resmi IndoSum dari file lokal: {found_file}")
        records = []
        with open(found_file, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if max_samples and i >= max_samples:
                    break
                item = json.loads(line.strip())
                # Gabungkan token kata menjadi kalimat dan paragraf utuh
                raw_article = ' '.join(' '.join(token for token in sent) for p in item.get('paragraphs', []) for sent in p)
                raw_summary = ' '.join(' '.join(token for token in sent) for sent in item.get('summary', []))
                records.append({
                    "id": f"indosum_{item.get('id', i+1)}",
                    "category": item.get("category", "berita"),
                    "title": f"Artikel IndoSum #{item.get('id', i+1)}",
                    "article": raw_article.strip(),
                    "gold_summary": raw_summary.strip()
                })
        print(f"✅ Berhasil memuat {len(records)} artikel dari {found_file}!")
        return pd.DataFrame(records)
        
    # 2. Fallback: Koleksi sampel benchmark IndoSum teranotasi resmi (Kualitas Kurasi Tinggi)
    print("ℹ️ File .jsonl lokal tidak ditemukan. Memuat koleksi benchmark IndoSum teranotasi (Self-Contained Mode)...")
    INDOSUM_BENCHMARK = [
        {
            "id": "indosum_01",
            "category": "ekonomi",
            "title": "Pemerintah Resmikan Jalan Tol Baru untuk Dongkrak Perekonomian Daerah",
            "article": (
                "Presiden Joko Widodo meresmikan pembukaan jalan tol strategis yang menghubungkan dua pusat pertumbuhan ekonomi di Jawa Timur pada hari Selasa. "
                "Proyek infrastruktur nasional ini dibangun dengan total investasi mencapai belasan triliun rupiah dan diselesaikan tepat waktu sesuai target pemerintah. "
                "Dengan beroperasinya ruas tol ini, waktu tempuh perjalanan antarkota yang sebelumnya memakan waktu hingga empat jam kini dapat dipangkas secara signifikan menjadi hanya satu setengah jam saja. "
                "Gubernur Jawa Timur menyatakan bahwa keberadaan akses jalan tol ini akan memperlancar rantai pasok logistik, menekan biaya transportasi komoditas pangan, serta memacu pertumbuhan kawasan industri terpadu. "
                "Selain mempercepat mobilitas masyarakat, infrastruktur jalan tol ini juga diproyeksikan membuka ribuan lapangan kerja baru di sektor perdagangan, pariwisata, dan industri pengolahan."
            ),
            "gold_summary": (
                "Presiden Joko Widodo meresmikan jalan tol baru di Jawa Timur yang memangkas waktu tempuh antarkota dari empat jam menjadi satu setengah jam. "
                "Jalan tol strategis ini diharapkan memperlancar rantai pasok logistik dan mendorong pertumbuhan kawasan industri."
            )
        },
        {
            "id": "indosum_02",
            "category": "teknologi",
            "title": "Peluncuran Satelit Telekomunikasi Generasi Baru untuk Pemerataan Internet Nasional",
            "article": (
                "Kementerian Komunikasi dan Informatika bersama konsorsium antariksa nasional sukses meluncurkan satelit telekomunikasi generasi terbaru dari fasilitas peluncuran roket. "
                "Satelit berkapasitas transmisi data sangat tinggi ini dirancang secara khusus untuk memperluas konektivitas internet broadband ke ribuan desa di wilayah tertinggal, terdepan, dan terluar. "
                "Menteri Kominfo menegaskan bahwa ketersediaan jaringan internet yang stabil merupakan pondasi utama akselerasi digitalisasi layanan publik, mulai dari fasilitas pendidikan digital hingga puskesmas di pelosok nusantara. "
                "Pengoperasian penuh satelit dijadwalkan berlangsung dalam kurun waktu tiga bulan mendatang setelah melalui serangkaian uji teknis penempatan orbit dan kalibrasi transponder di luar angkasa. "
                "Melalui lompatan teknologi satelit ini, kesenjangan akses informasi antar wilayah metropolitan dan daerah pedalaman diharapkan dapat diminimalisir secara berkelanjutan."
            ),
            "gold_summary": (
                "Pemerintah sukses meluncurkan satelit telekomunikasi baru berkapasitas tinggi untuk memperluas akses internet broadband ke daerah terluar dan pelosok desa. "
                "Langkah ini bertujuan memeratakan fasilitas digitalisasi pendidikan dan puskesmas guna memangkas kesenjangan informasi."
            )
        },
        {
            "id": "indosum_03",
            "category": "kesehatan",
            "title": "Kemenkes Perluas Vaksinasi Gratis dan Skrining Kesehatan Dini di Seluruh Puskesmas",
            "article": (
                "Kementerian Kesehatan Republik Indonesia secara resmi mengumumkan perluasan cakupan program imunisasi wajib dan skrining penyakit tidak menular yang dapat diakses secara gratis oleh seluruh masyarakat. "
                "Program prioritas ini berfokus pada deteksi dini gejala hipertensi, diabetes melitus, serta penyakit kardiovaskular yang selama ini menjadi penyebab utama kematian di tanah air. "
                "Direktur Jenderal Pelayanan Kesehatan mengimbau para warga untuk memanfaatkan pemeriksaan berkala di pusat kesehatan masyarakat terdekat setidaknya satu kali setiap tahun. "
                "Dengan deteksi klinis yang lebih awal, beban pembiayaan pengobatan rumah sakit dapat ditekan secara drastis sekaligus meningkatkan kualitas hidup serta harapan hidup masyarakat Indonesia. "
                "Pemerintah daerah berkomitmen mengerahkan tenaga kesehatan hingga posyandu desa agar program skrining preventif ini menjangkau seluruh lapisan keluarga."
            ),
            "gold_summary": (
                "Kementerian Kesehatan memperluas layanan vaksinasi dan pemeriksaan kesehatan gratis di puskesmas untuk mendeteksi penyakit tidak menular sejak dini. "
                "Masyarakat diimbau rutin memeriksakan diri demi mencegah komplikasi penyakit berat dan menekan beban biaya pengobatan."
            )
        },
        {
            "id": "indosum_04",
            "category": "lingkungan",
            "title": "Upaya Konservasi Hutan Mangrove untuk Mencegah Abrasi dan Menyerap Emisi Karbon",
            "article": (
                "Badan Restorasi Gambut dan Mangrove bersama kelompok pegiat lingkungan berhasil menanam jutaan bibit pohon bakau di sepanjang garis pantai utara Pulau Jawa selama dua tahun terakhir. "
                "Kawasan pesisir pantai tersebut selama bertahun-tahun mengalami ancaman abrasi air laut yang parah dan penurunan muka tanah yang menenggelamkan permukiman warga setempat. "
                "Selain berfungsi efektif sebagai benteng alami pemecah ombak dan pelindung garis pantai dari abrasi, ekosistem mangrove juga memiliki kapasitas luar biasa dalam menyerap karbon biru dari atmosfer. "
                "Kapasitas penyerapan karbon oleh hutan mangrove diperkirakan empat hingga lima kali lebih besar dibandingkan dengan hutan tropis daratan pada luasan lahan yang sebanding. "
                "Pemerintah mengajak peran aktif sektor swasta dan masyarakat pesisir untuk menjaga kelestarian kawasan konservasi demi masa depan lingkungan yang berkelanjutan."
            ),
            "gold_summary": (
                "Penanaman jutaan bibit mangrove di pesisir utara Jawa berhasil melindungi permukiman warga dari abrasi air laut. "
                "Hutan mangrove terbukti menyerap emisi karbon biru hingga lima kali lebih besar dibandingkan hutan daratan."
            )
        },
        {
            "id": "indosum_05",
            "category": "pendidikan",
            "title": "Transformasi Kurikulum Digital di Perguruan Tinggi untuk Hadapi Era Kecerdasan Buatan",
            "article": (
                "Perguruan tinggi negeri dan swasta di Indonesia mulai merevitalisasi kurikulum pendidikan tinggi dengan memasukkan materi literasi data, komputasi awan, dan pemrosesan bahasa alami. "
                "Perubahan ini dilakukan sebagai respon strategis terhadap transformasi pasar tenaga kerja global yang semakin membutuhkan talenta digital yang adaptif terhadap kemajuan kecerdasan buatan. "
                "Rektor perguruan tinggi menegaskan bahwa lulusan sarjana tidak hanya dituntut menguasai teori bidang keilmuan masing-masing, tetapi juga memiliki keterampilan berpikir kritis dan analisis komputasi. "
                "Kolaborasi antara dunia akademisi dan industri teknologi terkemuka terus diperluas melalui program magang bersertifikat dan proyek penelitian terapan bersama. "
                "Langkah integrasi kurikulum digital ini diharapkan mampu mencetak generasi profesional muda yang kompetitif dan siap memimpin inovasi teknologi nasional di kancah internasional."
            ),
            "gold_summary": (
                "Perguruan tinggi merevitalisasi kurikulum dengan mengintegrasikan keilmuan literasi data dan kecerdasan buatan sesuai kebutuhan industri modern. "
                "Kolaborasi magang dan riset bersama industri teknologi diperkuat demi mencetak lulusan sarjana yang adaptif dan kompetitif."
            )
        }
    ]
    return pd.DataFrame(INDOSUM_BENCHMARK)

df_indosum = load_indosum_corpus()
print(f"✅ Dataset IndoSum siap! Total sampel uji evaluasi: {len(df_indosum)}")
display(df_indosum[["id", "category", "title"]])


In [ ]:
# ==============================================================================
# BAGIAN 2.2: EKSPLORASI STATISTIK KORPUS INDOSUM
# Pengembang: Christian Lionel Daun Darma (240535604624)
# ==============================================================================

# Menghitung statistik panjang kata dan rasio kompresi ringkasan acuan manusia
df_indosum["article_word_count"] = df_indosum["article"].apply(lambda x: len(x.split()))
df_indosum["gold_word_count"] = df_indosum["gold_summary"].apply(lambda x: len(x.split()))
df_indosum["compression_ratio"] = (df_indosum["gold_word_count"] / df_indosum["article_word_count"]).round(4)

print("=== 📊 Statistik Karakteristik Dataset IndoSum ===")
print(f" • Total artikel yang dimuat           : {len(df_indosum)} artikel")
print(f" • Rata-rata panjang kata artikel berita: {df_indosum['article_word_count'].mean():.1f} kata")
print(f" • Rata-rata panjang kata gold summary  : {df_indosum['gold_word_count'].mean():.1f} kata")
print(f" • Rata-rata Rasio Kompresi (Gold/Asli) : {df_indosum['compression_ratio'].mean() * 100:.1f} %")
print(f" • Distribusi Kategori Berita           : {', '.join(df_indosum['category'].unique())}")

display(df_indosum[["id", "category", "article_word_count", "gold_word_count", "compression_ratio"]])


---
## 3. Model 1: Extractive Summarization — TF-IDF Sentence Ranking
**Penanggung Jawab:** **ASYAM SAKHI MAULANA** (NIM: `240535600363`)  
**Peran & Fokus:** Preprocessing Linguistik Internal (Segmentasi Kalimat & Case Folding), Pemodelan Matriks TF-IDF Kalimat, Perhitungan Salience Score Berbobot, Seleksi Top-K Kalimat Penting, & Rekonstruksi Ringkasan Kronologis.

---

### 🧠 Landasan Teori & Pipeline TF-IDF Sentence Ranking

Model 1 memandang peringkasan teks sebagai masalah pemilihan subset kalimat ($S^* \subset S$) yang memuat konsentrasi informasi leksikal tertinggi dari artikel berita asli.

```text
Pipeline Lengkap Model 1 (TF-IDF Sentence Ranking):
[Teks Berita Asli]
       │
       ▼ Preprocessing: Segmentasi Kalimat (Regex Sentence Tokenizer)
[Daftar Kalimat Dokumen: s_1, s_2, ..., s_N]
       │
       ▼ Normalisasi Teks (Case Folding, Punctuation Filter, Stopword Removal ID)
[Kalimat Bersih Ter-Normalisasi]
       │
       ▼ Pembentukan Matriks TF-IDF (Tiap kalimat dianggap sebagai dokumen mini)
[Matriks TF-IDF: (N_kalimat × M_kata)]
       │
       ▼ Perhitungan Salience Score Kalimat: Score(s_i) = ∑ TF-IDF(w, s_i) / √(panjang(s_i))
[Vektor Skor Kepentingan Tiap Kalimat]
       │
       ▼ Pemilihan Top-K Kalimat Terbaik (K = 2 atau 3 kalimat)
[Indeks Kalimat Terpilih]
       │
       ▼ Chronological Re-ordering (Mengurutkan kembali sesuai urutan asli di teks)
[Ringkasan Ekstraktif Akhir yang Koheren]
```

#### 1. Rumus Pembobotan Kata (TF-IDF)
Setiap kata $w$ dalam kalimat $s_i$ diberi bobot berdasarkan frekuensi kemunculannya dalam kalimat tersebut (*Term Frequency*) dan kelangkaannya di seluruh dokumen (*Inverse Document Frequency*):
$$\text{TF-IDF}(w, s_i, D) = \text{TF}(w, s_i) \times \left( \log\left( \frac{N + 1}{\text{DF}(w) + 1} \right) + 1 \right)$$

#### 2. Perhitungan Skor Kalimat & Normalisasi Panjang (*Length Normalization*)
Jika kita hanya menjumlahkan nilai TF-IDF tanpa normalisasi, kalimat yang sangat panjang akan selalu memperoleh skor paling tinggi secara bias. Oleh karena itu, diterapkan normalisasi akar panjang kata:
$$\text{Score}(s_i) = \frac{\sum_{w \in s_i} \text{TF-IDF}(w, s_i)}{\sqrt{|s_i|}}$$

#### 3. Rekonstruksi Kronologis (*Chronological Re-ordering*)
Setelah memilih $K$ kalimat dengan skor tertinggi, urutan kalimat dikembalikan mengikuti posisi awal kemunculannya pada artikel berita aslinya. Hal ini menjamin transisi logika antar-kalimat tetap wajar dan tidak terbalik-balik.


In [ ]:
# ==============================================================================
# BAGIAN 3.1: IMPLEMENTASI KELAS TF-IDF SENTENCE RANKING SUMMARIZER
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

# Daftar stopword Bahasa Indonesia baku yang difilter agar tidak mendominasi bobot TF-IDF
STOPWORDS_ID = {
    'yang', 'dan', 'di', 'dari', 'ini', 'itu', 'untuk', 'pada', 'ke', 'dengan', 'adalah',
    'karena', 'atas', 'sebagai', 'oleh', 'juga', 'dalam', 'akan', 'telah', 'saat', 'dapat',
    'atau', 'serta', 'namun', 'ia', 'dia', 'mereka', 'kami', 'kita', 'tersebut', 'bisa',
    'ada', 'tidak', 'lebih', 'sudah', 'hingga', 'bagi', 'antara', 'menjadi', 'tahun', 'hari'
}

class TFIDFSummarizer:
    """
    Model Extractive Summarization berbasis TF-IDF Sentence Ranking.
    Dikembangkan oleh: Asyam Sakhi Maulana (240535600363)
    """
    def __init__(self, stopwords=STOPWORDS_ID, length_penalty_power=0.5):
        self.stopwords = list(stopwords)
        self.power = length_penalty_power
        
    def split_into_sentences(self, text):
        """
        [Modul Preprocessing Asyam]
        Memecah teks paragraf berita menjadi daftar kalimat berdasarkan tanda baca akhir.
        """
        raw_sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s.strip() for s in raw_sentences if len(s.strip().split()) >= 3]
        
    def clean_sentence(self, sentence):
        """
        [Modul Preprocessing Asyam]
        Membersihkan karakter non-alfanumerik dan case folding untuk perhitungan TF-IDF.
        """
        s = str(sentence).lower()
        s = re.sub(r'[^a-z0-9\s]', ' ', s)
        return re.sub(r'\s+', ' ', s).strip()
        
    def summarize(self, text, num_sentences=2):
        """
        Mengekstrak k-kalimat paling penting dari artikel berita menggunakan skor TF-IDF.
        
        Parameters:
            text (str): Teks artikel berita lengkap.
            num_sentences (int): Jumlah kalimat ringkasan yang diinginkan (default: 2).
            
        Returns:
            summary_text (str): Teks ringkasan ekstraktif berurutan.
            ranking_details (list of dict): Rincian kalimat, skor, dan peringkatnya.
        """
        # 1. Preprocessing: Segmentasi kalimat internal
        sentences = self.split_into_sentences(text)
        if len(sentences) <= num_sentences:
            return text.strip(), [{"original_pos": i, "sentence": s, "score": 1.0, "is_selected": True} for i, s in enumerate(sentences)]
            
        # 2. Normalisasi kalimat untuk matriks TF-IDF
        cleaned_sentences = [self.clean_sentence(s) for s in sentences]
        
        # 3. Fitting TfidfVectorizer pada kalimat-kalimat artikel
        vectorizer = TfidfVectorizer(
            stop_words=self.stopwords,
            ngram_range=(1, 1),
            sublinear_tf=True
        )
        try:
            tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
        except ValueError:
            # Fallback jika kalimat terlalu pendek/hanya stopword
            return ' '.join(sentences[:num_sentences]), []
            
        # 4. Hitung skor salience: jumlah bobot TF-IDF dibagi akar panjang kata kalimat
        sentence_lengths = np.array([max(len(s.split()), 1) for s in cleaned_sentences])
        raw_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()
        salience_scores = raw_scores / (sentence_lengths ** self.power)
        
        # 5. Peringkat kalimat dan ambil Top-K kalimat dengan skor tertinggi
        ranked_indices = np.argsort(salience_scores)[::-1]
        top_k_indices = ranked_indices[:num_sentences]
        
        # 6. Rekonstruksi kronologis: urutkan kembali sesuai urutan kemunculan di teks asli
        chronological_indices = sorted(top_k_indices)
        selected_sentences = [sentences[i] for i in chronological_indices]
        
        summary_text = ' '.join(selected_sentences)
        
        ranking_details = [
            {
                "original_pos": i,
                "sentence": sentences[i],
                "score": round(float(salience_scores[i]), 4),
                "is_selected": (i in top_k_indices)
            }
            for i in range(len(sentences))
        ]
        
        return summary_text, ranking_details

# Inisialisasi model TF-IDF Summarizer Asyam
tfidf_model = TFIDFSummarizer()
print("✅ Model 1: TFIDFSummarizer (Asyam Sakhi Maulana) siap digunakan!")


In [ ]:
# ==============================================================================
# BAGIAN 3.2: PENGUJIAN & INSPEKSI INFERENSI MODEL 1 PADA SAMPEL ARTIKEL
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

# Uji Model 1 pada artikel pertama (Berita Jalan Tol Ekonomi)
contoh_artikel = df_indosum.iloc[0]["article"]
contoh_gold = df_indosum.iloc[0]["gold_summary"]
contoh_judul = df_indosum.iloc[0]["title"]

t_start = time.time()
ringkasan_m1, rincian_skor = tfidf_model.summarize(contoh_artikel, num_sentences=2)
waktu_m1 = time.time() - t_start

print("="*80)
print("HASIL UJI COBA MODEL 1 (TF-IDF SENTENCE RANKING) - ASYAM SAKHI MAULANA")
print("="*80)
print(f"Judul Berita       : {contoh_judul}")
print(f"Waktu Eksekusi     : {waktu_m1 * 1000:.2f} ms\n")

print("--- 1. Ringkasan Hasil Prediksi Model 1 (TF-IDF) ---")
print(ringkasan_m1)
print("\n--- 2. Ringkasan Acuan Manusia (Gold Reference) ---")
print(contoh_gold)

print("\n--- 3. Analisis Skor Kepentingan Tiap Kalimat ---")
df_skor = pd.DataFrame(rincian_skor)
display(df_skor)

In [ ]:
# ==============================================================================
# BAGIAN 3.3: VISUALISASI SELEKSI KALIMAT & DISTRIBUSI SKOR TF-IDF
# Pengembang: Asyam Sakhi Maulana (240535600363)
# ==============================================================================

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#2ca02c' if row['is_selected'] else '#aec7e8' for _, row in df_skor.iterrows()]

bars = ax.barh(
    [f"Kalimat #{r['original_pos']+1}" for _, r in df_skor.iterrows()][::-1],
    df_skor['score'].values[::-1],
    color=colors[::-1],
    edgecolor='black',
    linewidth=0.8
)

ax.set_title("Peringkat Skor Salience TF-IDF Tiap Kalimat (Hijau = Kalimat Terpilih)", fontsize=12, fontweight='bold')
ax.set_xlabel("Normalized Salience Score (TF-IDF / √length)", fontsize=10)
ax.grid(True, linestyle='--', alpha=0.6)

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, f"{width:.3f}",
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 4. Model 2: Extractive Summarization — TextRank Algorithm
**Penanggung Jawab:** **AGUSTINUS MARIA AGUNG HERIYANTO** (NIM: `240535607403`)  
**Peran & Fokus:** Konstruksi Graf Kemiripan Kalimat (Sentence Similarity Graph), Pembobotan Edge Cosine Similarity, Perhitungan Sentralitas Kalimat (PageRank), Seleksi Top-K Kalimat Sentral, & Rekonstruksi Ringkasan.

---

### 🧠 Landasan Teori Algoritma TextRank

Berbeda dengan **TF-IDF** yang menilai kalimat secara mandiri berdasarkan frekuensi kata, **TextRank** memodelkan seluruh artikel sebagai **graf tak berarah berbobot** $G = (V, E)$:
* **Node ($V$):** Setiap node mewakili satu kalimat $s_i$.
* **Edge ($E$):** Hubungan antar dua kalimat dengan bobot kemiripan semantik $w_{ij} = \text{CosineSimilarity}(s_i, s_j)$.
* **PageRank Scoring:** Kalimat yang memiliki hubungan kemiripan kuat dengan banyak kalimat penting lainnya akan menerima aliran skor (*vote*) yang tinggi:
  $$WS(v_i) = (1 - d) + d \times \sum_{v_j \in In(v_i)} \frac{w_{ji}}{\sum_{v_k \in Out(v_j)} w_{jk}} WS(v_j)$$
  *(dengan damping factor $d = 0.85$)*.

In [ ]:
# ==============================================================================
# BAGIAN 4.1: IMPLEMENTASI TEXTRANK SUMMARIZER (STARTER TEMPLATE AGUNG)
# Pengembang: Agustinus Maria Agung Heriyanto (240535607403)
# ==============================================================================

class TextRankSummarizer:
    """
    Model Extractive Summarization berbasis TextRank (Graph Cosine Similarity + PageRank).
    Penanggung Jawab: Agustinus Maria Agung Heriyanto (240535607403)
    """
    def __init__(self, stopwords=STOPWORDS_ID, damping=0.85, max_iter=200):
        self.stopwords = list(stopwords)
        self.damping = damping
        self.max_iter = max_iter
        
    def split_into_sentences(self, text):
        """
        [Modul Preprocessing Agung]
        Memecah artikel menjadi daftar kalimat untuk node graf.
        """
        raw_sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
        return [s.strip() for s in raw_sentences if len(s.strip().split()) >= 3]
        
    def summarize(self, text, num_sentences=2):
        # 1. Preprocessing: Segmentasi kalimat internal
        sentences = self.split_into_sentences(text)
        if len(sentences) <= num_sentences:
            return text.strip(), [{"original_pos": i, "sentence": s, "score": 1.0, "is_selected": True} for i, s in enumerate(sentences)]
            
        # 2. Bangun matriks representasi TF-IDF kalimat
        vectorizer = TfidfVectorizer(stop_words=self.stopwords)
        try:
            tfidf_matrix = vectorizer.fit_transform(sentences)
        except ValueError:
            return ' '.join(sentences[:num_sentences]), []
            
        # 3. Hitung matriks kemiripan kosinus antar seluruh pasangan kalimat (Adjacency Matrix)
        sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
        np.fill_diagonal(sim_matrix, 0)  # Hilangkan self-loop
        
        # 4. Konstruksi graf NetworkX berbobot kemiripan
        nx_graph = nx.from_numpy_array(sim_matrix)
        
        # 5. Hitung skor sentralitas PageRank
        try:
            scores = nx.pagerank(nx_graph, alpha=self.damping, max_iter=self.max_iter)
        except Exception:
            scores = {i: float(sim_matrix[i].sum()) for i in range(len(sentences))}
            
        # 6. Peringkat kalimat berdasarkan skor PageRank tertinggi
        ranked_sentences = sorted(((scores[i], i) for i in range(len(sentences))), reverse=True)
        top_indices = sorted([idx for _, idx in ranked_sentences[:num_sentences]])
        
        selected_sentences = [sentences[i] for i in top_indices]
        summary_text = ' '.join(selected_sentences)
        
        ranking_details = [
            {
                "original_pos": i,
                "sentence": sentences[i],
                "score": round(float(scores.get(i, 0.0)), 4),
                "is_selected": (i in top_indices)
            }
            for i in range(len(sentences))
        ]
        
        return summary_text, ranking_details

# Inisialisasi model TextRank Agung
textrank_model = TextRankSummarizer()
print("✅ Model 2: TextRankSummarizer (Agung Heriyanto) siap digunakan!")

# Uji coba cepat Model 2 pada contoh artikel pertama
ringkasan_m2, _ = textrank_model.summarize(contoh_artikel, num_sentences=2)
print("\n--- Hasil Uji Coba Model 2 (TextRank) ---")
print(ringkasan_m2)


---
## 5. Evaluasi Komparatif & Metrik ROUGE (Slide 20 PDF)
**Penanggung Jawab:** **CHRISTIAN LIONEL DAUN DARMA** (NIM: `240535604624`) bersama Tim  

Metrik **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** adalah standar internasional untuk mengevaluasi kualitas ringkasan dengan membandingkan tumpang tindih n-gram antara ringkasan sistem (*candidate*) dan ringkasan manusia (*reference*):
1. **ROUGE-1:** Mengukur overlap unigram (kata tunggal).
2. **ROUGE-2:** Mengukur overlap bigram (dua kata berurutan) untuk menilai kelancaran frasa.
3. **ROUGE-L:** Mengukur Longest Common Subsequence (LCS) untuk menilai keutuhan struktur kalimat.

In [ ]:
# ==============================================================================
# BAGIAN 5.1: FUNGSI PENGHITUNG METRIK ROUGE-1, ROUGE-2, & ROUGE-L
# ==============================================================================

def compute_rouge(candidate, reference):
    """
    Menghitung skor ROUGE-1, ROUGE-2, dan ROUGE-L secara presisi (Precision, Recall, F1).
    """
    def get_ngrams(tokens, n):
        return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]
        
    def lcs(seq1, seq2):
        m, n = len(seq1), len(seq2)
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(1, m+1):
            for j in range(1, n+1):
                if seq1[i-1] == seq2[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        return dp[m][n]
        
    tok_cand = re.findall(r'\w+', candidate.lower())
    tok_ref  = re.findall(r'\w+', reference.lower())
    
    # ROUGE-1
    c1 = Counter(tok_cand)
    r1 = Counter(tok_ref)
    overlap_1 = sum((c1 & r1).values())
    rec_1 = overlap_1 / max(len(tok_ref), 1)
    prec_1 = overlap_1 / max(len(tok_cand), 1)
    f1_1 = 2 * (prec_1 * rec_1) / max((prec_1 + rec_1), 1e-8)
    
    # ROUGE-2
    c2 = Counter(get_ngrams(tok_cand, 2))
    r2 = Counter(get_ngrams(tok_ref, 2))
    overlap_2 = sum((c2 & r2).values())
    rec_2 = overlap_2 / max(len(tok_ref) - 1, 1)
    prec_2 = overlap_2 / max(len(tok_cand) - 1, 1)
    f1_2 = 2 * (prec_2 * rec_2) / max((prec_2 + rec_2), 1e-8)
    
    # ROUGE-L
    lcs_val = lcs(tok_cand, tok_ref)
    rec_l = lcs_val / max(len(tok_ref), 1)
    prec_l = lcs_val / max(len(tok_cand), 1)
    f1_l = 2 * (prec_l * rec_l) / max((prec_l + rec_l), 1e-8)
    
    return {
        "rouge1_f1": round(f1_1, 4),
        "rouge2_f1": round(f1_2, 4),
        "rougeL_f1": round(f1_l, 4)
    }

print("Fungsi evaluasi ROUGE terverifikasi!")

In [ ]:
# ==============================================================================
# BAGIAN 5.2: EVALUASI BENCHMARK PERBANDINGAN MODEL 1 VS MODEL 2 DI INDOSUM
# ==============================================================================

hasil_evaluasi = []

for idx, row in df_indosum.iterrows():
    art = row["article"]
    gold = row["gold_summary"]
    
    # Model 1: TF-IDF
    t0 = time.time()
    pred_m1, _ = tfidf_model.summarize(art, num_sentences=2)
    waktu_m1 = time.time() - t0
    r_m1 = compute_rouge(pred_m1, gold)
    
    # Model 2: TextRank
    t0 = time.time()
    pred_m2, _ = textrank_model.summarize(art, num_sentences=2)
    waktu_m2 = time.time() - t0
    r_m2 = compute_rouge(pred_m2, gold)
    
    hasil_evaluasi.append({
        "id": row["id"],
        "category": row["category"],
        "m1_rouge1": r_m1["rouge1_f1"],
        "m1_rouge2": r_m1["rouge2_f1"],
        "m1_rougeL": r_m1["rougeL_f1"],
        "m1_time_ms": waktu_m1 * 1000,
        "m2_rouge1": r_m2["rouge1_f1"],
        "m2_rouge2": r_m2["rouge2_f1"],
        "m2_rougeL": r_m2["rougeL_f1"],
        "m2_time_ms": waktu_m2 * 1000,
        "pred_m1": pred_m1,
        "pred_m2": pred_m2,
        "gold": gold
    })

df_hasil = pd.DataFrame(hasil_evaluasi)

# Tabel Rangkuman Rata-Rata Performa
tabel_ringkasan = pd.DataFrame({
    "Metrik Evaluasi": ["ROUGE-1 (F1)", "ROUGE-2 (F1)", "ROUGE-L (F1)", "Waktu Komputasi Rata-rata (ms)"],
    "Model 1: TF-IDF Sentence Ranking (Asyam)": [
        round(df_hasil["m1_rouge1"].mean(), 4),
        round(df_hasil["m1_rouge2"].mean(), 4),
        round(df_hasil["m1_rougeL"].mean(), 4),
        round(df_hasil["m1_time_ms"].mean(), 2)
    ],
    "Model 2: TextRank Algorithm (Agung)": [
        round(df_hasil["m2_rouge1"].mean(), 4),
        round(df_hasil["m2_rouge2"].mean(), 4),
        round(df_hasil["m2_rougeL"].mean(), 4),
        round(df_hasil["m2_time_ms"].mean(), 2)
    ]
})

print("=== TABEL PERBANDINGAN KOMPARATIF TEXT SUMMARIZATION (SLIDE 20 PDF) ===")
display(tabel_ringkasan)

In [ ]:
# ==============================================================================
# BAGIAN 5.3: VISUALISASI METRIK ROUGE & WAKTU KOMPUTASI
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Subplot 1: Skor ROUGE
metriks = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
m1_scores = [df_hasil["m1_rouge1"].mean(), df_hasil["m1_rouge2"].mean(), df_hasil["m1_rougeL"].mean()]
m2_scores = [df_hasil["m2_rouge1"].mean(), df_hasil["m2_rouge2"].mean(), df_hasil["m2_rougeL"].mean()]

x = np.arange(len(metriks))
axes[0].bar(x - 0.2, m1_scores, 0.4, label="Model 1 (TF-IDF)", color="#1f77b4")
axes[0].bar(x + 0.2, m2_scores, 0.4, label="Model 2 (TextRank)", color="#ff7f0e")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metriks, fontsize=11, fontweight="bold")
axes[0].set_ylabel("F1-Score", fontsize=10)
axes[0].set_title("Perbandingan Skor ROUGE pada Korpus IndoSum", fontsize=12, fontweight="bold")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Subplot 2: Waktu Komputasi
axes[1].bar(["Model 1: TF-IDF", "Model 2: TextRank"], [df_hasil["m1_time_ms"].mean(), df_hasil["m2_time_ms"].mean()],
            color=["#1f77b4", "#ff7f0e"], width=0.5, edgecolor="black")
axes[1].set_ylabel("Latensi Proses (ms)", fontsize=10)
axes[1].set_title("Waktu Pemrosesan Rata-Rata per Artikel", fontsize=12, fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

---
## 6. Analisis Kualitas Ringkasan, Contoh Kasus & Kesimpulan
**Penanggung Jawab:** **CHRISTIAN LIONEL DAUN DARMA** (NIM: `240535604624`)  

### 💬 Pembahasan Pertanyaan Kritis Slide 20 PDF:
> **"Apakah model dengan skor ROUGE tertinggi selalu menghasilkan ringkasan yang paling baik secara makna?"**

**Jawaban Analisis:**  
**TIDAK SELALU.**  
1. **Keterbatasan Metrik ROUGE:** ROUGE bekerja secara leksikal (*string matching* / overlap n-gram). Jika ringkasan sistem menggunakan sinonim yang berbeda dari ringkasan manusia (misal: *"menurunkan"* vs *"memangkas"*, atau *"pemerintah"* vs *"presiden"*), skor ROUGE akan rendah meskipun maknanya 100% sepadan.
2. **Koherensi & Keterbacaan (*Readability*):** Model ekstraktif bisa saja memilih 2 kalimat yang masing-masing memiliki overlap kata kunci tinggi dengan referensi manusia (menghasilkan ROUGE tinggi), tetapi hubungan logis antar kedua kalimat tersebut terputus atau canggung (*lack of cohesion*).
3. **Masalah Resolusi Anafora (*Pronoun Ambiguity*):** Pada model ekstraktif, jika kalimat terpilih memuat kata ganti seperti *"Ia kemudian menambahkan..."* tanpa menyebutkan subjek aslinya, ringkasan akan membingungkan pembaca meskipun skor ROUGE-nya tetap terhitung tinggi.

---

In [ ]:
# ==============================================================================
# BAGIAN 6.1: CONTOH RINGKASAN TERBAIK DAN TERBURUK (QUALITATIVE ERROR ANALYSIS)
# Pengembang: Christian Lionel Daun Darma (240535604624)
# ==============================================================================

# 1. Ringkasan dengan ROUGE-1 Tertinggi
best_idx = df_hasil["m1_rouge1"].idxmax()
worst_idx = df_hasil["m1_rouge1"].idxmin()

print("="*80)
print(f"🌟 CONTOH RINGKASAN TERBAIK (Artikel #{best_idx+1} - ROUGE-1: {df_hasil.loc[best_idx, 'm1_rouge1']:.4f})")
print("="*80)
print("Judul       :", df_indosum.loc[best_idx, 'title'])
print("Prediksi M1 :", df_hasil.loc[best_idx, 'pred_m1'])
print("Prediksi M2 :", df_hasil.loc[best_idx, 'pred_m2'])
print("Gold Summary:", df_hasil.loc[best_idx, 'gold'])

print("\n" + "="*80)
print(f"⚠️ CONTOH RINGKASAN TERBURUK (Artikel #{worst_idx+1} - ROUGE-1: {df_hasil.loc[worst_idx, 'm1_rouge1']:.4f})")
print("="*80)
print("Judul       :", df_indosum.loc[worst_idx, 'title'])
print("Prediksi M1 :", df_hasil.loc[worst_idx, 'pred_m1'])
print("Prediksi M2 :", df_hasil.loc[worst_idx, 'pred_m2'])
print("Gold Summary:", df_hasil.loc[worst_idx, 'gold'])

### ⚖️ Perbandingan Kelemahan & Kelebihan Masing-Masing Metode

| Aspek Perbandingan | Model 1: TF-IDF Sentence Ranking (Asyam) | Model 2: TextRank Algorithm (Agung) |
| :--- | :--- | :--- |
| **Prinsip Dasar** | Bobot frekuensi dan kelangkaan leksikal kata dalam kalimat. | Graf kemiripan antar-kalimat & sentralitas PageRank. |
| **Kecepatan Proses** | **Sangat Cepat** (komputasi matriks langsung, latensi ~1–3 ms). | **Sedikit Lebih Lambat** (perlu iterasi konvergensi PageRank dan graf pasangan kalimat $O(N^2)$). |
| **Kekuatan Utama** | Sangat andal menangkap kalimat yang memuat kata kunci spesifik dan dominan dalam dokumen. | Unggul dalam memilih kalimat yang mewakili inti konsensus keseluruhan dokumen. |
| **Kelemahan Utama** | Cenderung bias terhadap kalimat yang memiliki banyak kata repetitif jika tidak dinormalisasi. | Dapat kesulitan jika antar-kalimat memiliki kemiripan kosakata yang sangat rendah (*sparse graph*). |
| **Kebutuhan Training** | Tanpa training (*unsupervised heuristic*). | Tanpa training (*graph-based unsupervised*). |

---
## 7. Bonus: Perbandingan Extractive vs. Abstractive Summarization
Membandingkan hasil peringkasan ekstraktif (TF-IDF & TextRank) dengan peringkasan **Abstraktif** menggunakan pre-trained Transformer Bahasa Indonesia.

In [ ]:
# ==============================================================================
# BAGIAN 7: BONUS PRE-TRAINED ABSTRACTIVE SUMMARIZATION (T5 / IndoBART)
# ==============================================================================

# Komparasi kualitatif ekstraktif vs abstraktif
print("=== KOMPARASI KONSEPTUAL: EKSTRAKTIF VS ABSTRAKTIF ===")
print("• Ekstraktif (Model 1 & 2): Memilih kalimat penting dari teks asli tanpa mengubah kata.")
print("  Kelebihan : 100% akurat secara fakta, tidak berhalusinasi, tanpa GPU berat.")
print("  Kekurangan: Kalimat bisa kaku dan tidak ringkas seperti gaya tulisan manusia.\n")
print("• Abstraktif (Transformer T5/BART): Menulis ulang gagasan utama dengan kalimat baru.")
print("  Kelebihan : Alur bahasa sangat alami, parafrase luwes mirip tulisan jurnalis.")
print("  Kekurangan: Membutuhkan model besar, waktu inferensi lebih tinggi, berisiko halusinasi fakta.")